# Mini batch training
In this notebook, we are first going to implement cross entropy, then move on to mini-batch training.

## Setup

In [ ]:
#| default_exp training

In [ ]:
#|export
import pickle,gzip,math,os,time,shutil,torch,matplotlib as mpl,numpy as np,matplotlib.pyplot as plt
from pathlib import Path
from torch import tensor,nn
import torch.nn.functional as F

In [ ]:
from fastcore.test import test_close

torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
torch.manual_seed(1)
mpl.rcParams['image.cmap'] = 'gray'

path_data = Path('data')
path_gz = path_data/'mnist.pkl.gz'
with gzip.open(path_gz, 'rb') as f: ((x_train, y_train), (x_valid, y_valid), _) = pickle.load(f, encoding='latin-1')
x_train, y_train, x_valid, y_valid = map(tensor, [x_train, y_train, x_valid, y_valid])

### Data

In [ ]:
num_input, input_dim = x_train.shape
c = y_train.max()+1
nh = 50

In [ ]:
class Model(nn.Module):
    def __init__(self, n_in, nh, n_out):
        super().__init__()
        self.layers = [nn.Linear(n_in,nh), nn.ReLU(), nn.Linear(nh,n_out)]
        
    def __call__(self, x):
        for l in self.layers: x = l(x)
        return x

In [ ]:
model = Model(input_dim, nh, c)
pred = model(x_train)
pred.shape

## Cross entropy loss
We implement the log of cross entropy loss (for easier computation).

In [ ]:
torch.sum(torch.exp(pred), dim=1, keepdim=True).shape

Broadcasting will now be automatically applied to divide `pred` with the sum of the exponent of each `x_j`, where `x_j` is the value for each of the class.

In [ ]:
def log_softmax(x):
    return x - x.exp().sum(-1, keepdim=True).log()

In [ ]:
log_softmax(pred)

In [ ]:
log_softmax(pred).shape

Next, we want to use the LogSumExp trick. The rationale is that we want the gradients to be as precise as possible (suppose we have a region where we would otherwise 'bounce around'). However, the exponent of a value may be very large, which makes the computed value less precise (because of how a computer can lose precision when the magnitude gets very large).

In [ ]:
m = pred.max(-1)[0]
m

In [ ]:
m[:, None].shape

In [ ]:
pred.shape

In [ ]:
def logsumexp_softmax(x):
    m = x.max(-1)[0]
    return (m + (x - m[:, None]).exp().sum(-1).log()).unsqueeze(-1)

In [ ]:
logsumexp_softmax(pred).shape

In [ ]:
logsumexp_softmax(pred)

In [ ]:
pred.exp().sum(-1, keepdim=True)

In [ ]:
def log_softmax(x):
    return x - logsumexp_softmax(x)

In [ ]:
log_softmax(pred)

Alright, our logsumexp trick is working correctly! Meanwhile, Pytorch actually already implements this for us.

In [ ]:
def log_softmax(x):
    return x - x.logsumexp(-1, keepdim=True)

In [ ]:
sm_pred = log_softmax(pred)
sm_pred

Now that we have calculated the softmax, we can proceed to calculate the cross entropy loss for some target $x$ and some prediction $p(x)$ by $- \sum x \log p(x) $.

However, given that our y values are 'one hot encoded' (in reality they represent an index of the predicted class), we can simplify the above equation to $ - \log(p_i)$, where $p_i$ is the probability of the actual class.

The above is also known as negative log likelihood loss.

In [ ]:
y_train

In [ ]:
sm_pred[torch.arange(sm_pred.shape[0]), y_train].shape, y_train.shape

In [ ]:
sm_pred[torch.arange(sm_pred.shape[0]), y_train].mean()

In [ ]:
def nll(sm_pred, tgt):
    return -1. * sm_pred[torch.arange(sm_pred.shape[0]), tgt].mean()

In [ ]:
nll(sm_pred, y_train)

In Pytorch, softmax and negative log likelihood loss are combined together in the softmax function.

In [ ]:
loss = nn.CrossEntropyLoss()
loss(pred, y_train)

## Basic training loop
A basic training loop repeats over the following steps:
- get the output of the model on a batch of inputs
- compare the output to the labels we have and compute a loss
- calculate the gradients of the loss with respect to every parameter of the model
- update said parameters with those gradients to make them a little bit better

In [ ]:
loss_function = F.cross_entropy

In [ ]:
bs = 50
xb = x_train[0:bs]
preds = model(xb)

In [ ]:
preds[0], preds.shape

In [ ]:
y_preds = preds.argmax(dim=1).float()
y_preds

In [ ]:
yb = y_train[0:bs]
yb

In [ ]:
loss_function(model(xb), yb)

In [ ]:
def accuracy(preds, actual): 
    return 1. * sum(preds.argmax(dim=1) == actual) / actual.shape[0]

In [ ]:
#|export
def accuracy(out, yb): return (out.argmax(dim=1)==yb).float().mean()

In [ ]:
accuracy(preds, yb)

In [ ]:
#|export
def report(loss, preds, yb): print(f'{loss:.2f}, {accuracy(preds, yb):.2f}')

In [ ]:
lr=0.5
epochs=3

In [ ]:
xb, yb = x_train[:bs], y_train[:bs]
y_pred = model(xb)
report(loss_function(y_pred, yb), y_pred, yb)

We can now write our training loop.

In [ ]:
for i in range(epochs):
    for b in range(0, num_input, bs):
        s = slice(b, min(b+bs, num_input))
        xb, yb = x_train[s], y_train[s]
        y_pred = model(xb)
        loss = loss_function(y_pred, yb)
        loss.backward()

        with torch.no_grad():
            for l in model.layers:
                if hasattr(l, 'weight'):
                    l.weight -= l.weight.grad * lr
                    l.bias -= l.bias.grad * lr
                    l.weight.grad.zero_()
                    l.bias.grad.zero_()

    report(loss, y_pred, yb)

We see that we are getting pretty good accuracy on our training set in just 3 epochs! Recall that we actually have 10 classes, so an accuracy of 96% on a balanced dataset is actually very good.

## Use parameters and optim

### Parameters
We can now use nn's parameters to help us reduce the number of lines of code. nn's parameters refer to the weights and biases of each layer.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, nh, output_dim):
        super().__init__()
        self.l1 = nn.Linear(input_dim, nh)
        self.l2 = nn.Linear(nh, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.l2(self.relu(self.l1(x)))

In [ ]:
model = MLP(input_dim, nh, c.item())

In [ ]:
list(model.parameters())

We see that calling `parameters()` on an object inheriting `nn.Module` gives us access to the list of weights and biases in that model.

We can now shorten our training loop.

In [ ]:
for i in range(epochs):
    for b in range(0, x_train.shape[0], bs):
        s = slice(b, min(x_train.shape[0], b+bs))
        xb, yb = x_train[s], y_train[s]
        y_pred = model(xb)
        loss = loss_function(y_pred, yb)
        loss.backward()

        with torch.no_grad():
            for p in model.parameters():
                p -= p.grad * lr
            model.zero_grad()

    report(loss, y_pred, yb)

We see that our code is now shorter. Next, we can use PyTorch's `optim` to make the code even shorter.

### Optim
PyTorch's `optim` offers various optimisation algorithms for training neural network, such as SGD, Adam etc. We will implement our own SGD optimizer from scratch.

In [ ]:
class SGDOptimizer():
    def __init__(self, params, lr=0.5):
        self.params = list(params)
        self.lr = lr

    def step(self):
        with torch.no_grad():
            for p in self.params:
                p -= p.grad * self.lr

    def zero_grad(self):
        for p in self.params:
            p.grad.zero_()

In [ ]:
model.parameters()

In [ ]:
model = nn.Sequential(nn.Linear(input_dim, nh), nn.ReLU(), nn.Linear(nh, c))
optim = SGDOptimizer(model.parameters())

We can further shorten our training loop.

In [ ]:
for i in range(epochs):
    for b in range(0, num_input, bs):
        s = slice(b, min(num_input, b+bs))
        xb, yb = x_train[s], y_train[s]
        y_pred = model(xb)
        loss = loss_function(y_pred, yb)
        loss.backward()
        optim.step()
        optim.zero_grad()
            
    report(loss, y_pred, yb)

Now that we have implemented SGD Optimizer from scratch, we can use PyTorch's `optim`.

In [ ]:
def get_model():
    model = nn.Sequential(nn.Linear(input_dim,nh), nn.ReLU(), nn.Linear(nh,10))
    return model, torch.optim.SGD(model.parameters(), lr=lr)

## Dataset and Dataloader
It is clunky to retrieve `xb`, `yb` separately in our code. We want to retrieve them together.

In [ ]:
#| export
class Dataset:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, i):
        return self.x[i], self.y[i]

In [ ]:
train_ds, valid_ds = Dataset(x_train, y_train), Dataset(x_valid, y_valid)
assert(len(train_ds) == len(x_train))
assert(len(valid_ds) == len(x_valid))

In [ ]:
xb, yb = train_ds[0:5]
assert xb.shape == (5, 28*28)
assert yb.shape == (5,)

Let's shorten our traning loop.

In [ ]:
model, optim = get_model()

In [ ]:
for i in range(epochs):
    for b in range(0, num_input, bs):
        s = slice(b, min(num_input, b+bs))
        xb, yb = train_ds[s]
        y_pred = model(xb)
        loss = loss_function(y_pred, yb)
        loss.backward()
        optim.step()
        optim.zero_grad()
            
    report(loss, y_pred, yb)

Our current training loop iterates over elements in the dataset as:
`for b in range(0, num_input, bs):
  s = slice(b, min(num_input, b+bs))
  xb, yb = train_ds[s]`

Let's replace this with a Dataloader.

In [ ]:
class Dataloader:
    def __init__(self, dataset, bs):
        self.dataset = dataset
        self.bs = bs

    # defines how an object of this class behaves in an iteration context
    def __iter__(self):
        for i in range(0, len(self.dataset), self.bs): yield self.dataset[i: i+bs]

In [ ]:
train_dl = Dataloader(train_ds, bs)
valid_dl = Dataloader(valid_ds, bs)

In [ ]:
model, optim = get_model()

In [ ]:
for i in range(epochs):
    for batch in train_dl:
        xb, yb = batch
        y_pred = model(xb)
        loss = loss_function(y_pred, yb)
        loss.backward()
        optim.step()
        optim.zero_grad()
            
    report(loss, y_pred, yb)

We have now implemented a Dataloader from scratch!

### Random sampling
We want our Dataloader to sample randomly during training (to smoothen out any potential bias in how the data is prepared), and sample non-randomly during validation.

In [ ]:
import random

In [ ]:
class Sampler():
    def __init__(self, ds, shuffle=False):
        self.n, self.shuffle = len(ds), shuffle

    def __iter__(self):
        res = list(range(self.n))
        if self.shuffle: random.shuffle(res)
        return iter(res)

In [ ]:
from itertools import islice

In [ ]:
ss = Sampler(train_ds)

In [ ]:
it = iter(ss)
for i in range(5): print(next(it))

In [ ]:
ss = Sampler(train_ds, shuffle=True)
it = iter(ss)
for i in range(5): print(next(it))

We are shuffling our dataset. Let's use the shuffled values as indices to select the data.

In [ ]:
import fastcore.all as fc

In [ ]:
??fc

In [ ]:
class BatchSampler():
    def __init__(self, sampler, bs, drop_last=False):
        fc.store_attr()

    def __iter__(self):
        yield from fc.chunked(iter(self.sampler), self.bs)

In [ ]:
batchs = BatchSampler(ss, 50)

In [ ]:
def collate(b):
    xs, ys = zip(*b)
    return torch.stack(xs), torch.stack(ys)

In [ ]:
class Dataloader():
    def __init__(self, ds, batchs, collate_fn=collate): fc.store_attr()

    def __iter__(self): yield from (self.collate_fn(self.ds[i] for i in b) for b in self.batchs)

In [ ]:
train_samp = BatchSampler(Sampler(train_ds, shuffle=True), bs)
val_samp = BatchSampler(Sampler(valid_ds), bs)

In [ ]:
train_dl = Dataloader(train_ds, batchs=train_samp)
val_dl = Dataloader(valid_ds, batchs=val_samp)

In [ ]:
model, opt = get_model()

In [ ]:
for i in range(epochs):
    num_train_rows = 0
    for batch in train_dl:
        xb, yb = batch
        num_train_rows += xb.shape[0]
        y_pred = model(xb)
        loss = loss_function(y_pred, yb)
        loss.backward()
        opt.step()
        opt.zero_grad()
            
    print("Num training rows in this epoch:", num_train_rows)
    report(loss, y_pred, yb)

### Multiprocessing Dataloader

In [ ]:
import torch.multiprocessing as mp
from fastcore.basics import store_attr

In [ ]:
train_ds[[3,6,8,1]]

In [ ]:
train_ds.__getitem__([3, 6, 8, 1])

We see that `__getitem__()` is the same as indexing into the tensor directly.

To translate this to multiple workers, we can use `map` to call `__getitem__()` on the corresponding index.

In [ ]:
for o in map(train_ds.__getitem__, ([3,6], [8,1])): print(o)

In [ ]:
class Dataloader():
    def __init__(self, ds, batchs, num_workers=1, collate_fn=collate):
        fc.store_attr()

    def __iter__(self):
        with mp.Pool(self.num_workers) as ex:
            print("started worker")
            yield from ex.map(self.ds.__getitem__, iter(self.batchs))

In [ ]:
train_dl = Dataloader(train_ds, batchs=train_samp, num_workers=2)
it = iter(train_dl)

In [ ]:
xb, yb = next(it)
xb.shape, yb.shape

For some reason each worker is facing problem with executing the mapping.

### PyTorch DataLoader
We can now use PyTorch's DataLoader.

In [ ]:
#| export
from torch.utils.data import DataLoader, SequentialSampler, RandomSampler, BatchSampler

In [ ]:
train_samp = BatchSampler(RandomSampler(train_ds), bs, drop_last=False)
valid_samp = BatchSampler(SequentialSampler(valid_ds), bs, drop_last=False)

In [ ]:
train_dl = DataLoader(train_ds, batch_sampler=train_samp, collate_fn=collate)
val_dl = DataLoader(valid_ds, batch_sampler=val_samp, collate_fn=collate)

In [ ]:
model, opt = get_model()
for i in range(epochs):
    for batch in train_dl:
        xb, yb = batch
        y_pred = model(xb)
        loss = loss_function(y_pred, yb)
        loss.backward()
        opt.step()
        opt.zero_grad()
            
    report(loss, y_pred, yb)

PyTorch can actually automatically generate the `BatchSampler` and `RandomSampler` for us.

In [ ]:
train_dl = DataLoader(train_ds, bs, shuffle=True, drop_last=True, num_workers=2)
val_dl = DataLoader(valid_ds, bs, shuffle=False, num_workers=2)

In [ ]:
model, opt = get_model()
for i in range(epochs):
    for batch in train_dl:
        xb, yb = batch
        y_pred = model(xb)
        loss = loss_function(y_pred, yb)
        loss.backward()
        opt.step()
        opt.zero_grad()
            
    report(loss, y_pred, yb)

## Validation
We check our model's performance on a validation set at the end of each epoch. We need to call `model.train()` before training, and `model.eval()` before inference 

In [ ]:
model, opt = get_model()

for i in range(epochs):
    model.train()
    for batch in train_dl:
        xb, yb = batch
        y_pred = model(xb)
        loss = loss_function(y_pred, yb)
        loss.backward()
        opt.step()
        opt.zero_grad()

    model.eval()
    val_loss = 0.
    val_acc = 0
    count = 0
    for batch in val_dl:
        xb, yb = batch
        n = len(xb)
        y_pred = model(xb)
        loss = loss_function(y_pred, yb)
        val_loss += loss.item() * n
        val_acc += accuracy(y_pred, yb).item() * n
        count += n

    print(i, val_loss/count, val_acc/count)

We have successfully implemented PyTorch's DataLoaders from scratch!

In [ ]:
#|export
def fit(epochs, model, loss_func, opt, train_dl, valid_dl):
    for epoch in range(epochs):
        model.train()
        for xb,yb in train_dl:
            loss = loss_func(model(xb), yb)
            loss.backward()
            opt.step()
            opt.zero_grad()

        model.eval()
        with torch.no_grad():
            tot_loss,tot_acc,count = 0.,0.,0
            for xb,yb in valid_dl:
                pred = model(xb)
                n = len(xb)
                count += n
                tot_loss += loss_func(pred,yb).item()*n
                tot_acc  += accuracy (pred,yb).item()*n
        print(epoch, tot_loss/count, tot_acc/count)
    return tot_loss/count, tot_acc/count

In [ ]:
#|export
def get_dls(train_ds, valid_ds, bs, **kwargs):
    return (DataLoader(train_ds, batch_size=bs, shuffle=True, **kwargs),
            DataLoader(valid_ds, batch_size=bs*2, **kwargs))

In [ ]:
import nbdev; nbdev.export.nb_export('04_minibatch_training.ipynb', lib_path="miniai")